In [8]:
# %%
import os
import glob
import yt_dlp

# --- Portable Directory Setup ---
# This looks at your current folder (scripts), goes up one level, then into research
OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "research", "youtube-transcripts"))

experts_queue = {
    "josh_braun": "https://www.youtube.com/@joshbraunsales",
    "jason_bay": "https://www.youtube.com/@jasondbay", 
    "john_barrows": "https://www.youtube.com/@JohnMBarrows",
    "will_allred": "https://www.youtube.com/@itslavenderduh",
    "jed_mahrle": "https://www.youtube.com/@practicalprospecting",
    "becc_holland": "https://www.youtube.com/@flipthescript-sales",
    "eric_nowoslawski": "https://www.youtube.com/@ericnowoslawski",
    "armand_farrokh": "https://www.youtube.com/@30MPC",

    # Guest Spot Video IDs (Takes only the first one)
    "charlotte_johnson": ["Z5mwjeM4YZA", "bq_SWgq9VUk"],
    "florin_tatulea": ["j2z-PISTO9c", "8Fo8nsfNZxQ"]
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Portable path detected: {OUTPUT_DIR}")

✅ Portable path detected: c:\Users\tkg\Documents\Projects\100hires-portfolio\research\youtube-transcripts


In [9]:
# %%
# THE UPGRADE: Clear out old markdown files before saving the new batch
print("Sweeping the directory for old data...")
old_files = glob.glob(os.path.join(OUTPUT_DIR, "*.md"))
for f in old_files:
    os.remove(f)

print(f"Cleaned {len(old_files)} files. Ready for V2.")

Sweeping the directory for old data...
Cleaned 0 files. Ready for V2.


In [10]:
# %%
transcripts_data = []

def process_single_video(expert, vid_url):
    ydl_opts = {
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['en'],
        'quiet': True,
        'no_warnings': True,
        'outtmpl': 'temp_transcript',
        'playlist_items': '1'  # Strictly limits channel scrapes to 1 video
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(vid_url, download=True)
            # Handle if result is a channel/playlist entries or single video
            entries = info.get('entries', [info])
            if not entries: return

            entry = entries[0]
            raw_date = entry.get('upload_date', '00000000')
            
            # Read and clean transcript
            transcript_text = "Transcript content not found."
            sub_files = glob.glob("temp_transcript*")
            if sub_files:
                with open(sub_files[0], 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    transcript_text = " ".join([l.strip() for l in lines if "-->" not in l and not l.strip().isdigit() and "WEBVTT" not in l])
                for f in sub_files: os.remove(f)

            transcripts_data.append({
                "expert": expert,
                "video_id": entry.get('id'),
                "title": entry.get('title'),
                "date": f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}",
                "url": f"https://www.youtube.com/watch?v={entry.get('id')}",
                "content": transcript_text
            })
            print(f"  ✓ Found: {entry.get('title')}")
        except Exception:
            print(f"  ✗ Failed: {vid_url}")

for expert, source in experts_queue.items():
    print(f"Processing {expert}...")
    target_url = f"https://www.youtube.com/watch?v={source[0]}" if isinstance(source, list) else f"{source}/videos"
    process_single_video(expert, target_url)

print(f"\nDone! {len(transcripts_data)} videos ready.")

Processing josh_braun...
  ✓ Found: How to Make a Cold Call                      
Processing jason_bay...
  ✓ Found: The Three Ts Cold Call Opener               
Processing john_barrows...
  ✓ Found: Decision Making - What's the Risk? | Sales Tips with John Barrows
Processing will_allred...
  ✓ Found: Cold Email 101: The Recap (Episode 11)       
Processing jed_mahrle...
  ✓ Found: Practical Prospecting Podcast Episode 4: 5 Ways to Find Your Competitor's Customers
Processing becc_holland...
  ✓ Found: How to Sell Without Selling | The Sidekick Life with Franchise Sidekick & Becc Holland
Processing eric_nowoslawski...
  ✓ Found: How to Use Hiring Signals for Cold Email in 2026
Processing armand_farrokh...
  ✓ Found: I Cold Called For 60 Minutes (And Went Slightly Unhinged)
Processing charlotte_johnson...
  ✓ Found: How to Write Cold Emails That Actually Get Replies | Lusha x 30MPC Webinar
Processing florin_tatulea...
  ✓ Found: Cold Email Masterclass: Frameworks That Actually Book Meeti

In [11]:
# %%
for data in transcripts_data:
    filename = os.path.join(OUTPUT_DIR, f"{data['expert']}_{data['video_id']}.md")
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# {data['title']}\n\n")
        f.write(f"**Expert:** {data['expert'].replace('_', ' ').title()}\n")
        f.write(f"**Upload Date:** {data['date']}\n")
        f.write(f"**Source URL:** {data['url']}\n\n")
        f.write("---\n\n")
        f.write("### Content\n\n")
        f.write(data['content'])

# Directory audit
print("Audit:")
for file in os.listdir(OUTPUT_DIR):
    if file.endswith(".md"):
        print(f"  - {file} ({os.path.getsize(os.path.join(OUTPUT_DIR, file))} bytes)")

Audit:
  - armand_farrokh_T73wbqNQiO0.md (91701 bytes)
  - becc_holland_y6CO23qfRDM.md (256603 bytes)
  - charlotte_johnson_Z5mwjeM4YZA.md (372303 bytes)
  - eric_nowoslawski_s_EUgLx-esY.md (54902 bytes)
  - florin_tatulea_j2z-PISTO9c.md (285604 bytes)
  - jason_bay_3I_6FL2IfLg.md (4644 bytes)
  - jed_mahrle_9BIrXMfOMHw.md (135821 bytes)
  - john_barrows_yo0uF68CeYc.md (33798 bytes)
  - josh_braun_hpXev0vUcvQ.md (13118 bytes)
  - will_allred_b0suhvh-s0I.md (8629 bytes)
